**Natural Language Processing and Text Analytics - Copenhagen Business School | Spring 2026**

# Mandatory Assignment 3
## Classifying AmItheAsshole: BoW, Local LLM, and OpenAI

<br>

***

# 0. Instructions

You will work with the [r/AmItheAsshole](https://www.reddit.com/r/AmItheAsshole/) dataset and build three classifiers:

1. **Task 1** — Bag of Words (BoW) with Logistic Regression
2. **Task 2** — A small LLM you run locally (Qwen2.5-0.5B-Instruct)
3. **Task 3** — An OpenAI LLM using course-provided credentials

<details>
<summary><b>About the AITA Dataset (click to expand)</b></summary>
<br>

![r/AmItheAsshole](https://drive.google.com/uc?export=view&id=11MVVxLuPxY9ZqNpC3ikMBkQov1MsrTOx)

This famous <a href="https://www.reddit.com/r/AmItheAsshole/">subreddit</a> describes itself as "A catharsis for the frustrated moral philosopher in all of us, and a place to finally find out if you were wrong in an argument that's been bothering you."

You post a description of a situation, and ask the community to judge whether you are in the wrong (YTA — "you're the asshole"), or not (NTA — "not the asshole"). We use only these two labels, giving us a binary classification task. (The dataset used in this assignment was obtained <a href="https://www.kaggle.com/datasets/jianloongliew/reddit/data">here</a>.)

See more info on the <a href="https://en.wikipedia.org/wiki/R/AmItheAsshole">AITA Wikipedia Page</a>.

Unlike topic classification tasks, AITA does not naturally lend itself to BoW approaches — the classification relies on understanding open-ended situation descriptions and applying moral principles, not just word frequencies.
</details>

# 1. Setup

## 1.1. Packages and imports

You need this API key: `OPENAI_NLP_API_KEY` (set it as a Colab secret or environment variable — do not paste into the notebook).

In [ ]:
!pip -q install openai transformers torch accelerate


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from openai import OpenAI
from transformers import pipeline
from tqdm.auto import tqdm


## 1.2. Course OpenAI credentials (for Task 3)

Add the course API key to Colab Secrets (the key icon in the left sidebar) with the name `OPENAI_NLP_API_KEY`.

In [ ]:
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_NLP_API_KEY")
    print("API key loaded from Colab secrets.")
except Exception:
    OPENAI_API_KEY = None
    print("Not running in Colab — set OPENAI_API_KEY manually.")

OPENAI_MODEL = "gpt-5-mini"


## 1.3. Load the AITA dataset

In [ ]:
def aita_url(size=500):
    file_mapping = {
        500:   "1maRVNpe09JLcqwmcHn6Xy-LVSb_RIPGv",
        1000:  "1kJOR2GRfHu6Hd1i3fTZG413tNABWaeG7",
        5000:  "1Xe7iNBEgkftpTfh1U1O_JKkCV7rUtKdg",
    }
    fid = file_mapping[size]
    return f"https://drive.google.com/uc?export=download&id={fid}"


def load_aita(url, test_size=0.2, random_state=42):
    df = (
        pd.read_csv(url)
        .dropna(subset=["text", "label"])
        .reset_index(drop=True)
        .astype({"label": str})
    )
    train, test = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["label"]
    )
    return train.reset_index(drop=True), test.reset_index(drop=True)


In [ ]:
# Use size=500 for Tasks 1 and 2. You can try size=1000 if you like.
# Do NOT use size=5000 for Task 2 (local LLM) — it will be very slow.
AITA_URL = aita_url(size=500)
train, test = load_aita(AITA_URL)

print(f"Train: {len(train)} rows | Test: {len(test)} rows")
print(train["label"].value_counts())
train.head(3)


***

# 2. Tasks

## Task 1: Bag of Words (BoW)

Use `CountVectorizer` to turn the post text into word-count features, then train a `LogisticRegression` classifier.

**Useful docs:**
- [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

In [ ]:
# TODO: Vectorize the training text, train a logistic regression, and evaluate on the test set.
# Report accuracy and print a classification_report.

# 1. Vectorize
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(train["text"])
X_test_bow  = vectorizer.transform(test["text"])

# 2. Train
# YOUR CODE HERE

# 3. Predict and evaluate
# YOUR CODE HERE


***

## Task 2: Small Local LLM (Qwen2.5-0.5B-Instruct)

Load a small Qwen model locally using HuggingFace `transformers` and use it to classify AITA posts.

The model (`Qwen/Qwen2.5-0.5B-Instruct`) is about 1 GB and can run on CPU, though a GPU will be much faster.

**Tip:** To keep run time reasonable, use the same 500-row dataset (or even a smaller slice of the test set).

In [ ]:
import torch

# Auto-detect the best available device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

# Available models (larger = better quality, slower and more memory):
#   "Qwen/Qwen2.5-0.5B-Instruct"  ~1 GB  (may collapse to one class)
#   "Qwen/Qwen2.5-1.5B-Instruct"  ~3 GB  (recommended)
local_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=device,
)


In [ ]:
SYSTEM_PROMPT = """You are a judge on the Reddit community r/AmItheAsshole.
Given a post, decide whether the poster is in the wrong.
Reply with exactly one word: either NTA or YTA."""


def predict_local(text):
    """Classify a single AITA post using the local Qwen model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    output = local_pipe(messages, max_new_tokens=10)[0]["generated_text"]
    reply = output[-1]["content"].strip().upper()
    if "YTA" in reply:
        return "YTA"
    return "NTA"


# TODO: Run predict_local on the test set and evaluate.
# Use tqdm to track progress, e.g.:
#   preds = [predict_local(text) for text in tqdm(test["text"], desc="Qwen")]
# YOUR CODE HERE


***

## Task 3: OpenAI LLM (course credentials)

Use the OpenAI API with the key provided by the course to classify AITA posts.

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)


In [ ]:
def predict_openai(text):
    """Classify a single AITA post using the course OpenAI model."""
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a judge on the Reddit community r/AmItheAsshole. "
                    "Given a post, decide whether the poster is in the wrong. "
                    "Reply with exactly one word: either NTA or YTA."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    reply = response.choices[0].message.content.strip().upper()
    if "YTA" in reply:
        return "YTA"
    return "NTA"


# TODO: Run predict_openai on the test set and evaluate.
# Use tqdm to track progress, e.g.:
#   preds = [predict_openai(text) for text in tqdm(test["text"], desc="OpenAI")]
# YOUR CODE HERE


***

# 3. Results and Reflections

## 3.1. Aggregate results

Collect the accuracy from each task and present them in a table.

In [ ]:
# TODO: Fill in your accuracy values from the three tasks above
results = pd.DataFrame([
    {"task": "Task 1 — BoW + Logistic Regression", "accuracy": None},
    {"task": "Task 2 — Local LLM (Qwen2.5-0.5B)",  "accuracy": None},
    {"task": "Task 3 — OpenAI LLM",                 "accuracy": None},
])
results


## 3.2. Reflections

### Q1: BoW for AITA
The AITA task is different from a topic-classification task like news categorisation. How well do you think a BoW approach is suited to AITA, and why? Does your result confirm your expectation?

*TODO: YOUR ANSWER HERE*

<br>

### Q2: Small LLM vs. large LLM
Compare Task 2 (small local model) and Task 3 (larger OpenAI model). What difference in performance did you observe, and what trade-offs exist between the two approaches (cost, speed, privacy, accuracy)?

*TODO: YOUR ANSWER HERE*

<br>

### Q3: Label quality
The AITA labels come from Reddit community votes. In what ways might these labels be noisy or debatable? Does this affect how we should interpret the accuracy scores?

*TODO: YOUR ANSWER HERE*